- ctrl + j 터미널에서 가상환경 만들기 → vscode에서 만들기
    * 가상환경 만들기 : python -m venv .venv
    * 가상환경 들어가기 : .venv\Scripts\activate
    * pip 업그레이드 : python -m pip install --upgrade pip
    * ipynb 파일 새로 만들기 : ex1_회귀모델 저장
    * pip install statsmodels joblib flask

    * pip freeze > requirement.txt
    * requirements.txt받은 측 : pip install -r requirements.txt

In [1]:
print(1)

1


- 국토교통부 실거래가 공걔 시스템 : https://rt.molit.go.kr/

In [2]:
import pandas as pd # csv 읽기
import statsmodels.api as sm # 회귀모델
import joblib #모델 저장

In [3]:
df = pd.read_csv('./data/trade_apt_api.csv', encoding='cp949')
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 318 entries, 0 to 317
Data columns (total 13 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   거래금액     318 non-null    int64  
 1   건축년도     318 non-null    int64  
 2   년        318 non-null    int64  
 3   법정동      318 non-null    object 
 4   아파트      318 non-null    object 
 5   월        318 non-null    int64  
 6   일        318 non-null    int64  
 7   전용면적     318 non-null    float64
 8   지번       318 non-null    object 
 9   지역코드     318 non-null    int64  
 10  층        318 non-null    int64  
 11  해제사유발생일  318 non-null    object 
 12  해제여부     318 non-null    object 
dtypes: float64(1), int64(7), object(5)
memory usage: 32.4+ KB


In [4]:
# 회귀 모델
X = df[['건축년도', '전용면적', '층']].copy()
X['const'] = 1
y = df['거래금액']
X.shape, y.shape

((318, 4), (318,))

In [5]:
model = sm.OLS(y, X).fit() # 회귀모뎅
model.summary()
# R-squared : X가 y를 설명해 주는  수치
# Adj. R-squared : 조정된 r squared
# Durbin-Watson : X끼리의 상관이 있는지 수치 (이상치는 2)
# coef(w와 b) : 

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:                   거래금액   R-squared:                       0.648
Model:                            OLS   Adj. R-squared:                  0.644
Method:                 Least Squares   F-statistic:                     192.4
Date:                Wed, 23 Sep 2026   Prob (F-statistic):           8.54e-71
Time:                        10:50:10   Log-Likelihood:                -3777.5
No. Observations:                 318   AIC:                             7563.
Df Residuals:                     314   BIC:                             7578.
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
==============================================================================
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
건축년도        1925.6916    212.616      9.057      0.000    1507.360    2344.023
전용면적         962.1507     47.367     20.313      0.000     868.955    1055.347
층           2058.1524    417.716      4.927      0.000    1236.276    2880.028
const      -3.855e+06   4.25e+05     -9.069      0.000   -4.69e+06   -3.02e+06
==============================================================================
Omnibus:                       20.985   Durbin-Watson:                   1.352
Prob(Omnibus):                  0.000   Jarque-Bera (JB):               42.734
Skew:                           0.345   Prob(JB):                     5.25e-10
Kurtosis:                       4.658   Cond. No.                     4.33e+05
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
[2] The condition number is large, 4.33e+05. This might indicate that there are
strong multicollinearity or other numerical problems.
"""

In [6]:
X.iloc[0]

건축년도     2002.00
전용면적       84.82
층           1.00
const       1.00
Name: 0, dtype: float64

In [7]:
# 모델 예측하기
format(round(model.predict([[2020, 84, 8, 1]])[0]*10000), ',')

'1,320,696,965'

In [9]:
# 모델 저장
import os
if not os.path.exists('model'): # model
    os.mkdir('model')
joblib.dump(model, './model/ex1_apt_price_regression.joblib')

['./model/ex1_apt_price_regression.joblib']

In [10]:
def predict_apt_price(year, square, floor):
    r_model = joblib.load('./model/ex1_apt_price_regression.joblib')
    input_data = [[int(year), int(square), int(floor), 1]]
    result = round(r_model.predict(input_data)[0]*10000)
    return format(result, ',') +'원'
predict_apt_price(2002, 102, 8)

'1,147,259,615원'